# NL2SHACL-Bench: End-to-End Tutorial

This notebook walks through the complete NL2SHACL-Bench pipeline, from raw SHACL shapes to a benchmark evaluation. It uses the example data in `examples/my-dataset/` and produces a dataset in the same format as `examples/example-nl2shacl-dataset/`.

The pipeline has two main stages:
- **Stage 1: Dataset Construction** — extract shape fragments, generate natural language descriptions, and produce a structured dataset.
- **Stage 2: Translation and Evaluation** — run a translation system on the dataset and evaluate the results.

---
**To use your own data**, replace the contents of `examples/my-dataset/shacl/` and `examples/my-dataset/ontology/` with your own files before running Stage 1.

## Prerequisites

In [ ]:
# Install required packages
!pip install rdflib pyshacl requests rdf-graph-gen google-genai 

  Using cached rdflib-7.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached pyshacl-0.31.0-py3-none-any.whl.metadata (37 kB)
  Using cached rdf_graph_gen-1.2.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached owlrl-7.1.4-py3-none-any.whl.metadata (3.8 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
  Using cached exrex-0.12.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached html5rdf-1.2.1-py2.py3-none-any.whl.metadata (7.5 kB)
Using cached rdflib-7.6.0-py3-none-any.whl (615 kB)
Using cached pyshacl-0.31.0-py3-none-any.whl (1.3 MB)
Using cached rdf_graph_gen-1.2.0-py3-none-any.whl (735 kB)
   ---------------------------------------- 0.0/950.8 kB ? eta -:--:--
   --------------------------------------- 950.8/950.8 kB 22.2 MB/s eta 0:00:00
Using cached e

In [2]:
import os

# Set paths
FRAMEWORK_ROOT = "."  # adjust if running from a different directory
EXAMPLE_DIR    = os.path.join(FRAMEWORK_ROOT, "examples", "my-dataset")
DATASET_DIR    = os.path.join(FRAMEWORK_ROOT, "examples", "example-nl2shacl-dataset")

print(f"Framework root : {os.path.abspath(FRAMEWORK_ROOT)}")
print(f"Example dir    : {os.path.abspath(EXAMPLE_DIR)}")
print(f"Dataset output : {os.path.abspath(DATASET_DIR)}")

Framework root : c:\Users\yuche\Desktop\NL2SHACL-Framework
Example dir    : c:\Users\yuche\Desktop\NL2SHACL-Framework\examples\my-dataset
Dataset output : c:\Users\yuche\Desktop\NL2SHACL-Framework\examples\example-nl2shacl-dataset


---
## Stage 1: Dataset Construction

### Step 1: Extract shape fragments

Extract top-level node shapes from the raw SHACL file. All `.ttl` files in `shacl/` are merged into a single graph before extraction, so cross-file references are resolved correctly.

In [3]:
!python Dataset-Construction/Data-Preprocessor/convert_shacl.py \
    --input-dir examples/my-dataset/shacl \
    --output-file examples/my-dataset/output_data.jsonl

INFO: Found 1 TTL file(s): ['shacl.ttl']
INFO: Parsing: shacl.ttl
INFO: Merged graph: 251 triples total
INFO:   NodeShapes: 27 | PropertyShapes: 2 | Auxiliary (inlined): 1 | Top-level: 27 (NodeShapes: 27, PropertyShapes with target: 0)
INFO: ============================================================
INFO: SUMMARY
INFO:   Files processed           : 1
INFO:   Top-level shape records   : 27
INFO:   Auxiliary shapes inlined  : 1
INFO:   sh:sparql triples stripped: 0
INFO:   Output JSONL records      : 27
INFO: ============================================================
INFO: Written to examples/my-dataset/output_data.jsonl


In [4]:
# Preview the first extracted record
import json

with open("examples/my-dataset/output_data.jsonl", encoding="utf-8") as f:
    first = json.loads(f.readline())

print(f"ID    : {first['id']}")
print(f"NL    : {first['nl']}")
print(f"SHACL :\n{first['shacl'][:300]}...")

ID    : meta:ApplicationComponentDomainShape
NL    : 
SHACL :
@prefix meta: <http://www.snik.eu/ontology/meta/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

meta:ApplicationComponentDomainShape a sh:NodeShape ;
    sh:class meta:ApplicationComponent ;
    sh:targetSubjectsOf meta:communicatesWith,
        meta:supports ....


### Step 2: Quality check and prefix extraction

Check structural completeness of extracted fragments and extract all prefix declarations. Review the log before proceeding — remove any `FAIL` entries from `output_data.jsonl` if needed.

In [5]:
!python Dataset-Construction/Data-Preprocessor/shacl_quality_check.py \
    examples/my-dataset/output_data.jsonl

Log written  → examples\my-dataset\output_data_check_log.txt
Prefixes     → examples\my-dataset\output_data_prefixes.json

Done. 27 entries — syntax 27✓/0✗ | structure 27✓/0⚠/0✗


In [6]:
# Print the last 20 lines of the check log
with open("examples/my-dataset/output_data_check_log.txt", encoding="utf-8") as f:
    lines = f.readlines()
print("".join(lines[-20:]))


──────────────────────────────────────────────────────────────────────
Entry : meta:isResponsibleForFunctionRangeShape  (line 26)
  [i]  Syntax        : PASS  (3 triples)
  [ii] Structure     : PASS
  [iv] Prefixes found: ['meta', 'sh']

──────────────────────────────────────────────────────────────────────
Entry : meta:supportsRangeShape  (line 27)
  [i]  Syntax        : PASS  (3 triples)
  [ii] Structure     : PASS
  [iv] Prefixes found: ['meta', 'sh']

SUMMARY
  Total entries processed      : 27
  (i)  Syntax   PASS / FAIL    : 27 / 0
  (ii) Structure PASS / WARN / FAIL : 27 / 0 / 0
  (iv) Unique prefix aliases   : 6
  ✓ No prefix alias collisions detected


### Step 3: Augment with ontology metadata

Look up ontology term metadata for each URI referenced in the shapes.

In [7]:
!python Dataset-Construction/Data-Preprocessor/ontology_augment.py \
    examples/my-dataset/output_data.jsonl \
    --prefixes examples/my-dataset/output_data_prefixes.json \
    --ontology-dir examples/my-dataset/ontology

  OK   ontology.ttl                                  +339 triples
Loaded 1 files (0 skipped) — 339 triples total, 62 subject URIs indexed
Augmented JSONL → examples\my-dataset\output_data_augmented.jsonl
Lookup log      → examples\my-dataset\output_data_lookup_log.txt

Done. 27 entries | domain: 31 found / 0 missing | value: 5 found / 0 no metadata


In [8]:
# Preview the ontology snippet added to the first record
with open("examples/my-dataset/output_data_augmented.jsonl", encoding="utf-8") as f:
    first_aug = json.loads(f.readline())

print(f"ID: {first_aug['id']}")
print(f"Ontology snippet ({len(first_aug['ontology_snippet'])} terms):")
for uri, meta in list(first_aug['ontology_snippet'].items())[:2]:
    print(f"  {uri}")
    print(f"    label: {meta.get('label', '—')}")

ID: meta:ApplicationComponentDomainShape
Ontology snippet (3 terms):
  http://www.snik.eu/ontology/meta/communicatesWith
    label: communicates with
  http://www.snik.eu/ontology/meta/supports
    label: supports


### Step 4: Generate description prompts

Generate LLM prompts for each shape fragment. Use `--subset` for a built-in domain role, or `--role` to provide a custom role sentence.

In [9]:
!python Dataset-Construction/Description-Generator/get_nl_prompt.py \
    --input examples/my-dataset/output_data_augmented.jsonl \
    --subset snik

Done. Written 27 records to examples/my-dataset/output_data_augmented_nl_prompts.jsonl


### Step 5: Call Gemini API to generate descriptions

> **Note:** This step requires a Gemini API key. Set the `GEMINI_API_KEY` environment variable before running.
> If you do not have an API key, skip this cell — the pre-generated file `examples/my-dataset/output_data_augmented_nl_prompts.jsonl` is already available and Step 6 will use it directly.

In [ ]:
# Skip this cell if you do not have a Gemini API key.
# Set your API key first:
# os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"

!python Dataset-Construction/Description-Generator/run_gemini.py \
    --input examples/my-dataset/output_data_augmented_nl_prompts.jsonl

^C


### Step 6: Human review

> **Note:** The Description Reviewer is a GUI tool and must be run outside this notebook:
> ```bash
> python Dataset-Construction/Description-Reviewer/UI_annotator.py
> ```
> For this tutorial, we use the pre-reviewed file already available in `examples/my-dataset/`.

In [17]:
# Preview a reviewed description
reviewed_path = "examples/my-dataset/output_data_description_reviewed.jsonl"

with open(reviewed_path, encoding="utf-8") as f:
    first_reviewed = json.loads(f.readline())

print(f"ID          : {first_reviewed['id']}")
print(f"Description : {first_reviewed['description']}")

ID          : meta:EntityTypeShape
Description : Every entity type must have at least one human-readable label and is strictly limited to a specific set of allowed attributes. It may include entity type components and indicate what it is based on. While no arbitrary information can be added, an entity type is also permitted to use standard top-level properties and specific connections to describe what it communicates with, the functions it supports, its associated software product, and its typical features.


### Step 7: Convert to dataset format

Combine the augmented JSONL and the reviewed descriptions into the final structured dataset format.

In [37]:
!python Dataset-Construction/convert_dataset.py \
    examples/my-dataset/output_data_augmented.jsonl \
    --descriptions examples/my-dataset/output_data_description_reviewed.jsonl \
    --out-dir examples/example-nl2shacl-dataset \
    --prefix example-nl2shacl

Loaded 8 reviewed descriptions from examples\my-dataset\output_data_description_reviewed.jsonl
Converted 8 of 27 entries to examples\example-nl2shacl-dataset/
  example-nl2shacl-descriptions.jsonl      (8 lines)
  example-nl2shacl-ontology_snippets.jsonl (8 lines)
  shacl/                                (8 .ttl files)

[WARN] 19 records skipped (missing fields):
  meta:ApplicationComponentDomainShape: missing description
  meta:ApprovesEntityTypeRangeShape: missing description
  meta:ComputerBasedApplicationComponentDomainShape: missing description
  meta:DecreasesRangeShape: missing description
  meta:EntityTypeComponentRangeShape: missing description
  meta:EntityTypeDomainShape: missing description
  meta:FunctionDomainShape: missing description
  meta:IncreasesRangeShape: missing description
  meta:IsBasedOnRangeShape: missing description
  meta:IsResponsibleForEntityTypeRangeShape: missing description
  meta:RoleDomainShape: missing description
  meta:RoleRangeShape: missing descr

In [38]:
# Verify the output structure
for fname in sorted(os.listdir("examples/example-nl2shacl-dataset")):
    print(fname)

shacl_files = os.listdir("examples/example-nl2shacl-dataset/shacl")
print(f"\nshacl/ contains {len(shacl_files)} .ttl files")

example-nl2shacl-descriptions.jsonl
example-nl2shacl-ontology_snippets.jsonl
shacl

shacl/ contains 8 .ttl files


---
## Stage 2: Translation and Evaluation

This stage evaluates a translation system on the dataset produced in Stage 1. We use the minimal rule-based translator as a reference example. To evaluate your own system, follow `NL2SHACL-Translator/TRANSLATOR_SPEC.md`.

### Step 8: Run the translator

The rule-based translator reads the dataset and produces translated SHACL shapes. It requires no API access.

In [39]:
!python NL2SHACL-Translator/translator_example_rule_based.py \
    --subset example-nl2shacl-dataset \
    --dataset-dir examples

Written 8 records to your_outputs.jsonl

Next step: attach reference SHACL and run evaluation:
  python NL2SHACL-Translator/attach_reference.py \
      --input your_outputs.jsonl \
      --dataset-dir examples \
      --output processed-output/example-nl2shacl-dataset_rule_based_processed.jsonl


In [40]:
# Preview the translator output
with open("your_outputs.jsonl", encoding="utf-8") as f:
    first_output = json.loads(f.readline())

print(f"ID           : {first_output['id']}")
print(f"Output SHACL :\n{first_output.get('output_shacl', '(null)')[:300]}")

ID           : example-nl2shacl-1
Output SHACL :
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:meta:ApplicationComponentShape
    a sh:NodeShape ;
    sh:targetClass <http://www.snik.eu/ontology/meta/Appl


### Step 9: Attach reference SHACL

Attach the reference shapes from the dataset to each translated record.

In [43]:
os.makedirs("processed-output", exist_ok=True)

!python NL2SHACL-Translator/attach_reference.py \
    --input your_outputs.jsonl \
    --dataset-dir examples \
    --subset example-nl2shacl-dataset \
    --output processed-output/example-nl2shacl-dataset_rule_based_processed.jsonl

Written 8 records to processed-output/example-nl2shacl-dataset_rule_based_processed.jsonl


### Step 10: Run evaluation

In [47]:
!python Shapes-Evaluation/run_evaluation.py \
    --input processed-output\example-nl2shacl-dataset_rule_based_processed.jsonl

True
220
Mode: all evaluators

Input:   processed-output\example-nl2shacl-dataset_rule_based_processed.jsonl
Output:  evaluation-output\example-nl2shacl-dataset_rule_based_eval.jsonl
Subset:  example-nl2shacl-dataset
Model:   rule_based
Running: ['semantic', 'structural', 'validity']
Normalize RDF lists: False
Total records: 8

[1/8] Processing 'example-nl2shacl-1'...
[5/8] Processing 'example-nl2shacl-5'...

Done.
  Evaluated: 8
  Skipped:   0
  Validity failed: 0
  Output: evaluation-output\example-nl2shacl-dataset_rule_based_eval.jsonl

All files processed.


  - SEMANTIC: [example-nl2shacl-4] Data graph generation from GT SHACL failed: StopIteration: 
  - SEMANTIC: [example-nl2shacl-5] Data graph generation from GT SHACL failed: StopIteration: 
  - SEMANTIC: [example-nl2shacl-6] Data graph generation from GT SHACL failed: StopIteration: 
  - SEMANTIC: [example-nl2shacl-8] Data graph generation from GT SHACL failed: StopIteration: 


### Step 11: Compute metrics

In [48]:
!python Shapes-Evaluation/compute_metrics.py \
    --input evaluation-output/

Found 1 file(s).

File: example-nl2shacl-dataset_rule_based_eval.jsonl
  Log saved to: eval-logs\example-nl2shacl-dataset_rule_based_eval_log.txt

  Subset       example-nl2shacl
  Model        dataset_rule_based
  N            8  (N_valid=8, N_sem=4)
  ────────────────────────────────────────────────────────────
  RDF-VR                         1.0000    RDF-ER:   0.0000
  Spec-VR                        1.0000    Spec-ER:  0.0000
  Vocab-VR                       1.0000    Vocab-ER: 0.0000
  ────────────────────────────────────────────────────────────
  EMR                            0.0000
  PMS                            0.1557
  ────────────────────────────────────────────────────────────
  SER                            0.5000

Metrics summary saved to: metrics-output\metrics_summary.csv
Done.


In [49]:
# Display the metrics summary
import csv

with open("metrics-output/metrics_summary.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for row in rows:
    print(row)

{'subset': 'example-nl2shacl', 'model': 'dataset_rule_based', 'N': '8', 'N_valid': '8', 'N_semantic_valid': '4', 'RDF_VR': '1.0', 'RDF_ER': '0.0', 'Spec_VR': '1.0', 'Spec_ER': '0.0', 'Vocab_VR': '1.0', 'Vocab_ER': '0.0', 'EMR': '0.0', 'PMS': '0.155746336996337', 'SER': '0.5'}


---
## Summary

| Step | Script | Input | Output |
|------|--------|-------|--------|
| 1 | `convert_shacl.py` | `shacl/` | `output_data.jsonl` |
| 2 | `shacl_quality_check.py` | `output_data.jsonl` | `output_data_check_log.txt`, `output_data_prefixes.json` |
| 3 | `ontology_augment.py` | `output_data.jsonl` | `output_data_augmented.jsonl` |
| 4 | `get_nl_prompt.py` | `output_data_augmented.jsonl` | `output_data_augmented_nl_prompts.jsonl` |
| 5 | `run_gemini.py` | `output_data_augmented_nl_prompts.jsonl` | `..._generated_description.jsonl` |
| 6 | `UI_annotator.py` | generated descriptions | `output_data_description_reviewed.jsonl` |
| 7 | `convert_dataset.py` | augmented + reviewed | `snik-descriptions.jsonl`, `snik-ontology_snippets.jsonl`, `shacl/` |
| 8 | `translator_example_rule_based.py` | dataset | `your_outputs.jsonl` |
| 9 | `attach_reference.py` | `your_outputs.jsonl` | `processed-output/xxx_processed.jsonl` |
| 10 | `run_evaluation.py` | `processed-output/` | `evaluation-output/xxx_eval.jsonl` |
| 11 | `compute_metrics.py` | `evaluation-output/` | `metrics-output/metrics_summary.csv` |